In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/house-prices-advanced-regression-techniques/test.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Load the train and test datasets
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

In [3]:
# Extract the target variable SalePrice from the train dataset
y_train = train['SalePrice']

In [4]:
# Drop the target variable from both the train and test datasets
X_train = train.drop(['Id', 'SalePrice'], axis=1)
X_test = test.drop(['Id'], axis=1)

In [5]:
# Fill missing values with the mean of the column
X_train = X_train.fillna(X_train.mean())
X_test = X_test.fillna(X_test.mean())

/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:2: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  
/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:3: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  This is separate from the ipykernel package so we can avoid doing imports until


In [6]:
# One-hot encode the categorical features
categorical_cols = X_train.select_dtypes(include=['object']).columns
encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
X_train_encoded = encoder.fit_transform(X_train[categorical_cols])
X_test_encoded = encoder.transform(X_test[categorical_cols])

In [7]:
# Create new feature dataframes with encoded and numeric columns
X_train_new = pd.concat([X_train.select_dtypes(include=['int64', 'float64']), pd.DataFrame(X_train_encoded)], axis=1)
X_test_new = pd.concat([X_test.select_dtypes(include=['int64', 'float64']), pd.DataFrame(X_test_encoded)], axis=1)

In [8]:
# Transform the features into polynomial features
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train_new)
X_test_poly = poly.transform(X_test_new)

/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:1692: FutureWarning: Feature names only support names that are all strings. Got feature names with dtypes: ['int', 'str']. An error will be raised in 1.2.
  FutureWarning,
/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:1692: FutureWarning: Feature names only support names that are all strings. Got feature names with dtypes: ['int', 'str']. An error will be raised in 1.2.
  FutureWarning,
/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:1692: FutureWarning: Feature names only support names that are all strings. Got feature names with dtypes: ['int', 'str']. An error will be raised in 1.2.
  FutureWarning,


In [9]:
# Split the train dataset into train and validation sets
X_train_poly, X_val_poly, y_train, y_val = train_test_split(X_train_poly, y_train, test_size=0.2, random_state=42)

In [10]:
# Train a linear regression model on the polynomial features
model = LinearRegression()
model.fit(X_train_poly, y_train)

LinearRegression()

In [11]:
# Make predictions on the validation set and calculate the root mean squared error (RMSE)
y_pred = model.predict(X_val_poly)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f'RMSE on validation set: {rmse:.2f}')

RMSE on validation set: 102674.91


In [12]:
# Use the trained model to make predictions on the test set
test_pred = model.predict(X_test_poly)

In [13]:
# Save the predictions to a CSV file and submit to Kaggle
submission = pd.DataFrame({'Id': test['Id'], 'SalePrice': test_pred})
submission.to_csv('submission.csv', index=False)